<a href="https://colab.research.google.com/github/ranobirRoy/mindSync01/blob/main/Random_Forest_StudentsLife_Dataset_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "kagglehub[pandas-datasets]" pandas openpyxl pyarrow

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "dartweichen/student-life"
)

print("Dataset downloaded at:", dataset_path)

100%|██████████| 390M/390M [00:04<00:00, 88.7MB/s]

Extracting files...


Dataset downloaded at: /root/.cache/kagglehub/datasets/dartweichen/student-life/versions/1


In [ ]:
import glob
import json
import os
import pandas as pd

# ১. মূল dataset ফোল্ডারের সঠিক পাথ খুঁজে বের করা
base_dir = dataset_path
for root, dirs, files in os.walk(dataset_path):
  if "sensing" in dirs and "EMA" in dirs:
    base_dir = root
    break
print("Base Directory:", base_dir)

# ২. EMA_definition.json থেকে Stress, Sleep, Mood, PAM এর আসল প্রশ্ন ও সংজ্ঞা দেখা
ema_def_path = os.path.join(base_dir, "EMA", "EMA_definition.json")
if os.path.exists(ema_def_path):
  with open(ema_def_path, "r", encoding="utf-8") as f:
    ema_defs = json.load(f)
  print("\n--- EMA Definitions (Target Candidates) ---")
  if isinstance(ema_defs, list):
    for item in ema_defs:
      if item.get("name") in ["Stress", "Sleep", "Mood", "PAM", "Social"]:
        print(json.dumps(item, indent=2))
  elif isinstance(ema_defs, dict):
    for k, v in ema_defs.items():
      if k in ["Stress", "Sleep", "Mood", "PAM", "Social"]:
        print(k, ":", json.dumps(v, indent=2))
else:
  print("\nEMA_definition.json পাওয়া যায়নি।")

# ৩. Stress, Sleep, Mood, PAM ফোল্ডারের আসল JSON ফাইলের ভেতরের নমুনা এবং ইউজার সংখ্যা দেখা
print("\n--- EMA Response Samples & Active Users ---")
for ema_type in ["Stress", "Sleep", "Mood", "PAM"]:
  folder = os.path.join(base_dir, "EMA", "response", ema_type)
  if os.path.exists(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.json")))
    non_empty_users = 0
    total_records = 0
    sample_record = None
    for file in files:
      try:
        with open(file, "r", encoding="utf-8") as f:
          data = json.load(f)
        if isinstance(data, list) and len(data) > 0:
          non_empty_users += 1
          total_records += len(data)
          if sample_record is None:
            # এমন একটি রেকর্ড নেওয়া যাতে null ছাড়াও আসল উত্তর আছে
            for rec in data:
              if len(rec.keys()) > 2:
                sample_record = rec
                break
            if sample_record is None:
              sample_record = data[0]
      except Exception as e:
        pass
    print(
        f"[{ema_type}] Total Files: {len(files)} | Users with data:"
        f" {non_empty_users} | Total Records: {total_records}"
    )
    print(f"   Sample Record: {sample_record}")

# ৪. Survey (PHQ-9 এবং PerceivedStressScale) এর গঠন, ইউজার সংখ্যা এবং উত্তরের ধরন দেখা
print("\n--- Survey Data Check ---")
for surv in ["PHQ-9.csv", "PerceivedStressScale.csv"]:
  s_path = os.path.join(base_dir, "survey", surv)
  if os.path.exists(s_path):
    df_s = pd.read_csv(s_path)
    print(
        f"\n[{surv}] Shape: {df_s.shape} | Unique Users:"
        f" {df_s['uid'].nunique()}"
    )
    if "type" in df_s.columns:
      print("Pre/Post counts:\n", df_s["type"].value_counts(dropna=False))
    print("Columns:", list(df_s.columns[:5]), "...")
    # প্রথম প্রশ্নটির কী কী ধরনের উত্তর আছে তা দেখা
    first_q_col = [c for c in df_s.columns if c not in ["uid", "type"]][0]
    print(
        f"Unique values in first question ('{first_q_col}'):"
        f" {df_s[first_q_col].unique()}"
    )

Base Directory: /root/.cache/kagglehub/datasets/dartweichen/student-life/versions/1/dataset

--- EMA Definitions (Target Candidates) ---
{
  "name": "Social",
  "questions": [
    {
      "options": "[1]0-4 persons, [2]5-9 persons, [3]10-19 persons, [4]20-49 persons, [5]50-99 persons, [6]over 100 persons, ",
      "question_id": "number",
      "question_text": "How many people did you have contact with yesterday, including anyone you said hello to, chatted, talked or discussed matters with, whether you did it face-to-face, by telephone, by mail or on the internet, and whether you personally knew the person or not? Please select one of the following categories that best matches your estimate:"
    },
    {
      "options": "",
      "question_id": "location",
      "question_text": ""
    }
  ]
}
{
  "name": "Stress",
  "questions": [
    {
      "options": "[1]A little stressed, [2]Definitely stressed, [3]Stressed out, [4]Feeling good, [5]Feeling great, ",
      "question_id": "level"

In [ ]:
import glob
import json
import os
import pandas as pd

# ১. সব ইউজারের Stress EMA ডেটা একত্রিত করা
stress_folder = os.path.join(base_dir, "EMA", "response", "Stress")
stress_files = sorted(glob.glob(os.path.join(stress_folder, "Stress_*.json")))

records = []
for fpath in stress_files:
  uid = os.path.basename(fpath).replace("Stress_", "").replace(".json", "")
  with open(fpath, "r", encoding="utf-8") as f:
    data = json.load(f)
  for item in data:
    # কখনো উত্তর 'level' key-তে থাকে, কখনো বা 'null' key-তে চলে যায় কি না তাও চেক করা
    lvl = item.get("level")
    ts = item.get("resp_time")
    if lvl is not None and ts is not None:
      records.append({"uid": uid, "resp_time": int(ts), "level": str(lvl)})

df_stress = pd.DataFrame(records)

# ২. টাইমস্ট্যাম্পকে স্থানীয় সময়ে রূপান্তর (Dartmouth College: US/Eastern timezone)
df_stress["datetime"] = pd.to_datetime(
    df_stress["resp_time"], unit="s", utc=True
).dt.tz_convert("America/New_York")
df_stress["date"] = df_stress["datetime"].dt.date
df_stress["level_num"] = pd.to_numeric(df_stress["level"], errors="coerce")
df_stress = df_stress.dropna(subset=["level_num"])
df_stress["level_num"] = df_stress["level_num"].astype(int)

print("Total valid Stress responses:", len(df_stress))
print("Unique users with valid Stress responses:", df_stress["uid"].nunique())

# ৩. প্রতিটি অপশনের (1 থেকে 5) সংখ্যা দেখা
level_map = {
    1: "1: A little stressed",
    2: "2: Definitely stressed",
    3: "3: Stressed out",
    4: "4: Feeling good",
    5: "5: Feeling great",
}
print("\n--- Raw Stress Level Distribution ---")
for k, v in level_map.items():
  cnt = (df_stress["level_num"] == k).sum()
  pct = (cnt / len(df_stress)) * 100
  print(f"{v:25s}: {cnt:4d} ({pct:.1f}%)")


# ৪. User-Day (প্রতি মানুষ, প্রতি দিন) অনুযায়ী সাজালে কতগুলো স্যাম্পল হয় তা পরীক্ষা করা
# অপশন ক: 1, 2, 3 = Stressed (1) বনাম 4, 5 = Feeling Good/Great (0)
# অপশন খ: 2, 3 = High Stress (1) বনাম 1, 4, 5 = Low/No Stress (0)
def check_daily_balance(df, stressed_levels, label_name):
  df_temp = df.copy()
  df_temp["is_stressed"] = df_temp["level_num"].isin(stressed_levels).astype(int)
  # যদি দিনে একাধিকবার উত্তর দেয়, তবে ওই দিনের গড় বা সর্বোচ্চ অবস্থা দেখা
  daily = (
      df_temp.groupby(["uid", "date"])
      .agg(
          mean_stress_bin=("is_stressed", "mean"),
          any_stressed=("is_stressed", "max"),
          resp_count=("is_stressed", "count"),
      )
      .reset_index()
  )
  # Majority vote বা >= 0.5 হলে ওই দিনটিকে Stressed (1) ধরা
  daily["target_majority"] = (daily["mean_stress_bin"] >= 0.5).astype(int)

  print(f"\n--- Daily Level Check ({label_name}) ---")
  print(
      f"Total User-Days: {len(daily)} | Unique Users: {daily['uid'].nunique()}"
  )
  vc = daily["target_majority"].value_counts()
  for cls in [0, 1]:
    c = vc.get(cls, 0)
    print(f"Class {cls}: {c} ({c/len(daily)*100:.1f}%)")


check_daily_balance(
    df_stress, [1, 2, 3], "Scheme A: [1,2,3]=Stressed vs [4,5]=Good"
)
check_daily_balance(
    df_stress,
    [2, 3],
    "Scheme B: [2,3]=Definitely/Stressed Out vs [1,4,5]=Little/Good",
)

# ৫. প্রতি ইউজারের কত দিন করে ডেটা আছে তার সারাংশ
user_days = df_stress.groupby("uid")["date"].nunique()
print("\n--- User-Days per Student Summary ---")
print(user_days.describe())
print("Users with >= 10 days of data:", (user_days >= 10).sum())
print("Users with < 5 days of data:", (user_days < 5).sum())

Total valid Stress responses: 2167
Unique users with valid Stress responses: 48

--- Raw Stress Level Distribution ---
1: A little stressed     :  974 (44.9%)
2: Definitely stressed   :  347 (16.0%)
3: Stressed out          :  266 (12.3%)
4: Feeling good          :  457 (21.1%)
5: Feeling great         :  123 (5.7%)

--- Daily Level Check (Scheme A: [1,2,3]=Stressed vs [4,5]=Good) ---
Total User-Days: 1270 | Unique Users: 48
Class 0: 263 (20.7%)
Class 1: 1007 (79.3%)

--- Daily Level Check (Scheme B: [2,3]=Definitely/Stressed Out vs [1,4,5]=Little/Good) ---
Total User-Days: 1270 | Unique Users: 48
Class 0: 838 (66.0%)
Class 1: 432 (34.0%)

--- User-Days per Student Summary ---
count    48.000000
mean     26.458333
std      15.176912
min       3.000000
25%      16.000000
50%      24.500000
75%      37.500000
max      71.000000
Name: date, dtype: float64
Users with >= 10 days of data: 41
Users with < 5 days of data: 2


In [ ]:
import glob
import os
import pandas as pd

sensing_folders = [
    "activity",
    "conversation",
    "phonelock",
    "dark",
    "phonecharge",
]

print("--- Sensing Folders & Column Check ---")
for s_type in sensing_folders:
  folder_path = os.path.join(base_dir, "sensing", s_type)
  if os.path.exists(folder_path):
    files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
    if len(files) > 0:
      df_sample = pd.read_csv(files[0], nrows=3)
      print(
          f"\n[{s_type}] Total Files: {len(files)} | Sample File:"
          f" {os.path.basename(files[0])}"
      )
      print("   Exact Columns:", list(df_sample.columns))
      print("   First Row:", df_sample.iloc[0].to_dict())
  else:
    print(f"\n[{s_type}] Folder NOT found!")

--- Sensing Folders & Column Check ---

[activity] Total Files: 49 | Sample File: activity_u00.csv
   Exact Columns: ['timestamp', ' activity inference']
   First Row: {'timestamp': 1364356801, ' activity inference': 0}

[conversation] Total Files: 49 | Sample File: conversation_u00.csv
   Exact Columns: ['start_timestamp', ' end_timestamp']
   First Row: {'start_timestamp': 1364359600, ' end_timestamp': 1364359812}

[phonelock] Total Files: 49 | Sample File: phonelock_u00.csv
   Exact Columns: ['start', 'end']
   First Row: {'start': 1364358992, 'end': 1364364807}

[dark] Total Files: 49 | Sample File: dark_u00.csv
   Exact Columns: ['start', 'end']
   First Row: {'start': 1364365266, 'end': 1364385049}

[phonecharge] Total Files: 49 | Sample File: phonecharge_u00.csv
   Exact Columns: ['start', 'end']
   First Row: {'start': 1364358885, 'end': 1364364806}


In [ ]:
import glob
import json
import os
import numpy as np
import pandas as pd

print("Feature Engineering শুরু হচ্ছে... অনুগ্রহ করে অপেক্ষা করুন...")

# ১. Target Label (Scheme B) তৈরি করা (1,270 User-Days)
stress_folder = os.path.join(base_dir, "EMA", "response", "Stress")
stress_files = sorted(glob.glob(os.path.join(stress_folder, "Stress_*.json")))

stress_records = []
for fpath in stress_files:
  uid = os.path.basename(fpath).replace("Stress_", "").replace(".json", "")
  with open(fpath, "r", encoding="utf-8") as f:
    data = json.load(f)
  for item in data:
    lvl = item.get("level")
    ts = item.get("resp_time")
    if lvl is not None and ts is not None:
      stress_records.append(
          {"uid": uid, "resp_time": int(ts), "level": str(lvl)}
      )

df_stress = pd.DataFrame(stress_records)
df_stress["datetime"] = pd.to_datetime(
    df_stress["resp_time"], unit="s", utc=True
).dt.tz_convert("America/New_York")
df_stress["date"] = df_stress["datetime"].dt.date
df_stress["level_num"] = pd.to_numeric(df_stress["level"], errors="coerce")
df_stress = df_stress.dropna(subset=["level_num"])
df_stress["level_num"] = df_stress["level_num"].astype(int)

# Scheme B: [2, 3] = 1 (Elevated Stress), [1, 4, 5] = 0 (Baseline/Normal)
df_stress["is_stressed"] = df_stress["level_num"].isin([2, 3]).astype(int)

df_target = (
    df_stress.groupby(["uid", "date"])
    .agg(
        stress_ratio=("is_stressed", "mean"),
        ema_resp_count=("is_stressed", "count"),
    )
    .reset_index()
)
df_target["target"] = (df_target["stress_ratio"] >= 0.5).astype(int)


# ২. Interval Sensor (conversation, phonelock, dark, phonecharge) প্রসেস করার ফাংশন
def extract_interval_features(folder_name, prefix, start_col, end_col):
  folder_path = os.path.join(base_dir, "sensing", folder_name)
  files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
  daily_list = []

  for fpath in files:
    uid = (
        os.path.basename(fpath)
        .replace(f"{folder_name}_", "")
        .replace(".csv", "")
    )
    df = pd.read_csv(fpath)
    df.columns = df.columns.str.strip()  # নামের শুরুর বা শেষের স্পেস মুছে ফেলা

    if len(df) == 0:
      continue

    df["duration_min"] = (df[end_col] - df[start_col]) / 60.0
    # অস্বাভাবিক ঋণাত্মক বা ২৪ ঘণ্টার বেশি (১৪৪০ মিনিট) রেকর্ড ফিল্টার করা
    df = df[(df["duration_min"] > 0) & (df["duration_min"] <= 1440)].copy()

    df["date"] = (
        pd.to_datetime(df[start_col], unit="s", utc=True)
        .dt.tz_convert("America/New_York")
        .dt.date
    )

    agg = (
        df.groupby("date")
        .agg(
            total_min=("duration_min", "sum"),
            mean_min=("duration_min", "mean"),
            std_min=("duration_min", "std"),
            max_min=("duration_min", "max"),
            count=("duration_min", "count"),
        )
        .reset_index()
    )
    agg["std_min"] = agg["std_min"].fillna(0)
    agg["uid"] = uid

    agg = agg.rename(
        columns={
            "total_min": f"{prefix}_total_min",
            "mean_min": f"{prefix}_mean_min",
            "std_min": f"{prefix}_std_min",
            "max_min": f"{prefix}_max_min",
            "count": f"{prefix}_count",
        }
    )
    daily_list.append(agg)

  return pd.concat(daily_list, ignore_index=True)


print("১/৩: Interval sensors (conversation, lock, dark, charge) প্রসেস হচ্ছে...")
df_conv = extract_interval_features(
    "conversation", "conv", "start_timestamp", "end_timestamp"
)
df_lock = extract_interval_features("phonelock", "lock", "start", "end")
df_dark = extract_interval_features("dark", "dark", "start", "end")
df_charge = extract_interval_features("phonecharge", "charge", "start", "end")

# ৩. Activity Sensor প্রসেস করা
print("২/৩: Activity sensor প্রসেস হচ্ছে...")
act_folder = os.path.join(base_dir, "sensing", "activity")
act_files = sorted(glob.glob(os.path.join(act_folder, "*.csv")))
act_list = []

for fpath in act_files:
  uid = os.path.basename(fpath).replace("activity_", "").replace(".csv", "")
  df = pd.read_csv(fpath)
  df.columns = df.columns.str.strip()
  if len(df) == 0:
    continue

  df["date"] = (
      pd.to_datetime(df["timestamp"], unit="s", utc=True)
      .dt.tz_convert("America/New_York")
      .dt.date
  )
  # 0: Stationary, 1: Walking, 2: Running, 3: Unknown
  df["is_stat"] = (df["activity inference"] == 0).astype(int)
  df["is_walk"] = (df["activity inference"] == 1).astype(int)
  df["is_run"] = (df["activity inference"] == 2).astype(int)
  df["is_active"] = df["activity inference"].isin([1, 2]).astype(int)

  agg = (
      df.groupby("date")
      .agg(
          act_stationary_ratio=("is_stat", "mean"),
          act_walking_ratio=("is_walk", "mean"),
          act_running_ratio=("is_run", "mean"),
          act_active_ratio=("is_active", "mean"),
          act_samples=("is_stat", "count"),
      )
      .reset_index()
  )
  agg["uid"] = uid
  act_list.append(agg)

df_act = pd.concat(act_list, ignore_index=True)

# ৪. Sleep EMA ফোল্ডার থেকে ঘুমের ঘণ্টা ও মান বের করা
print("৩/৩: Sleep EMA এবং সব ফিচার একসাথে Merge করা হচ্ছে...")
sleep_folder = os.path.join(base_dir, "EMA", "response", "Sleep")
sleep_files = sorted(glob.glob(os.path.join(sleep_folder, "Sleep_*.json")))
sleep_records = []
for fpath in sleep_files:
  uid = os.path.basename(fpath).replace("Sleep_", "").replace(".json", "")
  with open(fpath, "r", encoding="utf-8") as f:
    data = json.load(f)
  for item in data:
    ts = item.get("resp_time")
    hr = item.get("hour")
    rt = item.get("rate")
    if ts is not None and (hr is not None or rt is not None):
      sleep_records.append({
          "uid": uid,
          "resp_time": int(ts),
          "sleep_hour_code": pd.to_numeric(hr, errors="coerce"),
          "sleep_rate": pd.to_numeric(rt, errors="coerce"),
      })
df_sleep = pd.DataFrame(sleep_records)
df_sleep["date"] = (
    pd.to_datetime(df_sleep["resp_time"], unit="s", utc=True)
    .dt.tz_convert("America/New_York")
    .dt.date
)
df_sleep_daily = (
    df_sleep.groupby(["uid", "date"])
    .agg(
        sleep_hour_code=("sleep_hour_code", "mean"),
        sleep_rate=("sleep_rate", "mean"),
    )
    .reset_index()
)

# ৫. Target DataFrame এর সাথে সব সেন্সর ফিচার Merge করা
df_master = df_target.copy()
for df_feat in [df_act, df_conv, df_lock, df_dark, df_charge, df_sleep_daily]:
  df_master = pd.merge(df_master, df_feat, on=["uid", "date"], how="left")

# ৬. Personal Baseline (ব্যক্তির নিজস্ব গড় আচরণ থেকে প্রতিদিনের পার্থক্য) তৈরি করা
for col in [
    "act_active_ratio",
    "conv_total_min",
    "lock_total_min",
    "dark_total_min",
]:
  user_mean = df_master.groupby("uid")[col].transform("mean")
  df_master[f"{col}_dev"] = df_master[col] - user_mean

# ৭. ফলাফল ও সারাংশ প্রিন্ট করা
feature_cols = [
    c
    for c in df_master.columns
    if c not in ["uid", "date", "stress_ratio", "target"]
]
print("\n=== Feature Matrix Construction Complete ===")
print(f"Final Dataset Shape: {df_master.shape}")
print(f"Total Samples (User-Days): {len(df_master)}")
print(f"Unique Users: {df_master['uid'].nunique()}")
print(f"Total Features Extracted: {len(feature_cols)}")
print("\nTarget Class Balance:")
print(df_master["target"].value_counts())

print("\nMissing Value Percentage per Feature Group (Top 10):")
missing_pct = (df_master[feature_cols].isna().sum() / len(df_master) * 100).round(1)
print(missing_pct.sort_values(ascending=False).head(10))

Feature Engineering শুরু হচ্ছে... অনুগ্রহ করে অপেক্ষা করুন...
১/৩: Interval sensors (conversation, lock, dark, charge) প্রসেস হচ্ছে...
২/৩: Activity sensor প্রসেস হচ্ছে...
৩/৩: Sleep EMA এবং সব ফিচার একসাথে Merge করা হচ্ছে...

=== Feature Matrix Construction Complete ===
Final Dataset Shape: (1270, 36)
Total Samples (User-Days): 1270
Unique Users: 48
Total Features Extracted: 32

Target Class Balance:
target
0    838
1    432
Name: count, dtype: int64

Missing Value Percentage per Feature Group (Top 10):
sleep_rate          31.7
sleep_hour_code     31.7
charge_std_min      25.0
charge_total_min    25.0
charge_count        25.0
charge_mean_min     25.0
charge_max_min      25.0
dark_total_min      13.3
dark_max_min        13.3
dark_std_min        13.3
dtype: float64


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ১. Data Leakage এড়াতে শুধুমাত্র খাঁটি ডেইলি সেন্সর ও EMA ফিচারগুলো নির্বাচন করা (_dev ফিচার বাদ দিয়ে)
pure_feature_cols = [
    c
    for c in df_master.columns
    if c not in ["uid", "date", "stress_ratio", "target"]
    and not c.endswith("_dev")
]

X = df_master[pure_feature_cols].copy()
y = df_master["target"].values
groups = df_master["uid"].values

print(f"Total Pure Features Selected: {len(pure_feature_cols)}")
print("Features:", pure_feature_cols)

# ২. প্রথম মডেল পাইপলাইন তৈরি (Imputation -> Scaling -> Random Forest)
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

# ৩. Leave-One-Subject-Out (LOSO) Cross-Validation চালানো
logo = LeaveOneGroupOut()

y_true_all = []
y_pred_all = []
y_prob_all = []
subject_metrics = []

print(
    "\nLOSO Cross-Validation শুরু হচ্ছে (৪৮ জন ইউজারের জন্য ৪৮টি ফোল্ড)..."
)

for train_idx, test_idx in logo.split(X, y, groups=groups):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y[train_idx], y[test_idx]
  test_uid = groups[test_idx][0]

  # মডেল ট্রেন করা (শুধুমাত্র ৪৭ জন ইউজারের ডেটায়)
  rf_pipeline.fit(X_train, y_train)

  # ৪৮তম ইউজারের (অদেখা ইউজার) ওপর প্রেডিকশন করা
  y_pred = rf_pipeline.predict(X_test)
  y_prob = rf_pipeline.predict_proba(X_test)[:, 1]

  y_true_all.extend(y_test)
  y_pred_all.extend(y_pred)
  y_prob_all.extend(y_prob)

  subject_metrics.append({
      "uid": test_uid,
      "n_days": len(y_test),
      "acc": accuracy_score(y_test, y_pred),
  })

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
y_prob_all = np.array(y_prob_all)

# ৪. সামগ্রিক পারফরম্যান্স (Pooled LOSO Metrics) প্রিন্ট করা
acc = accuracy_score(y_true_all, y_pred_all)
prec = precision_score(y_true_all, y_pred_all, zero_division=0)
rec = recall_score(y_true_all, y_pred_all, zero_division=0)
f1 = f1_score(y_true_all, y_pred_all, zero_division=0)
macro_f1 = f1_score(y_true_all, y_pred_all, average="macro", zero_division=0)
auc = roc_auc_score(y_true_all, y_prob_all)

print("\n=== First Model (Random Forest) LOSO Results ===")
print(f"Overall Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision (Cls 1): {prec:.4f}")
print(f"Recall (Cls 1)   : {rec:.4f}")
print(f"F1-Score (Cls 1) : {f1:.4f}")
print(f"Macro F1-Score   : {macro_f1:.4f}")
print(f"ROC-AUC Score    : {auc:.4f}")

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_true_all, y_pred_all)
print(
    pd.DataFrame(
        cm,
        index=["Actual 0 (Normal)", "Actual 1 (Stressed)"],
        columns=["Pred 0 (Normal)", "Pred 1 (Stressed)"],
    )
)

print("\n--- Classification Report ---")
print(
    classification_report(
        y_true_all,
        y_pred_all,
        target_names=["Class 0 (Normal)", "Class 1 (Stressed)"],
    )
)

df_sub = pd.DataFrame(subject_metrics)
print(
    f"Subject-wise Mean Accuracy across 48 users: {df_sub['acc'].mean():.4f} ±"
    f" {df_sub['acc'].std():.4f}"
)

Total Pure Features Selected: 28
Features: ['ema_resp_count', 'act_stationary_ratio', 'act_walking_ratio', 'act_running_ratio', 'act_active_ratio', 'act_samples', 'conv_total_min', 'conv_mean_min', 'conv_std_min', 'conv_max_min', 'conv_count', 'lock_total_min', 'lock_mean_min', 'lock_std_min', 'lock_max_min', 'lock_count', 'dark_total_min', 'dark_mean_min', 'dark_std_min', 'dark_max_min', 'dark_count', 'charge_total_min', 'charge_mean_min', 'charge_std_min', 'charge_max_min', 'charge_count', 'sleep_hour_code', 'sleep_rate']

LOSO Cross-Validation শুরু হচ্ছে (৪৮ জন ইউজারের জন্য ৪৮টি ফোল্ড)...

=== First Model (Random Forest) LOSO Results ===
Overall Accuracy : 0.6039 (60.39%)
Precision (Cls 1): 0.4223
Recall (Cls 1)   : 0.4468
F1-Score (Cls 1) : 0.4342
Macro F1-Score   : 0.5648
ROC-AUC Score    : 0.5781

--- Confusion Matrix ---
                     Pred 0 (Normal)  Pred 1 (Stressed)
Actual 0 (Normal)                574                264
Actual 1 (Stressed)              239            